# Notebook 0: Generate Fake Data for SFT Pipeline Testing

Use GPT-OSS-120B to generate 10 synthetic outbound sales conversations,
then convert them directly into training examples that notebook 02 can consume.
This lets you test the full pipeline (00 → 02) without waiting for real data.

**Output:** `data/train_examples.jsonl` (same format notebook 01 would produce)

In [ ]:
import torch
import json
import re
import random
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ============================================================
# Configuration — update MODEL_DIR before running
# ============================================================
MODEL_DIR = "/path/to/gpt-oss-120b"   # <-- FILL IN: local checkpoint directory
OUTPUT_DIR = Path("data")
OUTPUT_FILE = OUTPUT_DIR / "train_examples.jsonl"  # feeds directly into notebook 02
NUM_CONVERSATIONS = 10

# Preprocessing config (same as notebook 01)
MIN_AGENT_TOKENS = 15
CONTEXT_POPULATE_RATE = 0.15
WINDOW_SIZE_K = 20

## Load Model (4-bit quantized for inference)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded from {MODEL_DIR}")
print(f"Vocab size: {len(tokenizer)}, Device map: {model.hf_device_map}")

## Generation Prompt Template\n\nThe prompt instructs the model to produce a transcript in our raw format.\nWe vary the target turn count per conversation to get diverse lengths.

In [ ]:
GENERATION_PROMPT_TEMPLATE = """\
Generate a realistic outbound sales call transcript between an American Express \
sales agent and a potential business customer. The agent is calling to pitch a \
corporate card product.

Requirements:
- The conversation should have approximately {num_turns} total lines (alternating agent and customer).
- Agent always speaks first.
- Use this EXACT format for each line — one per line, no blank lines between turns:
agent(u1): [agent's words]
customer(u2): [customer's words]
agent(u3): [agent's words]
customer(u4): [customer's words]
...and so on, strictly alternating, incrementing the utterance number each line.
- Make it realistic: include greetings, needs discovery, objection handling, product discussion.
- This call ended with the customer agreeing to learn more or schedule a follow-up (converted).
- Do NOT include any text before or after the transcript lines.
- Do NOT include blank lines between turns.

Transcript:
agent(u1):"""

# Scenario hints to inject variety across conversations
SCENARIOS = [
    "The customer is a small business owner who currently uses a competitor's card.",
    "The customer is a CFO at a mid-size company evaluating expense management tools.",
    "The customer is a startup founder skeptical about annual fees.",
    "The customer is a procurement manager who needs to consolidate travel spending.",
    "The customer is a restaurant owner interested in cash-back rewards.",
    "The customer just expanded to a second office and needs more employee cards.",
    "The customer is price-sensitive and keeps asking about fees.",
    "The customer is interested but says they need to discuss with a partner first.",
    "The customer had a bad experience with Amex years ago and is hesitant.",
    "The customer is very busy and keeps trying to end the call quickly.",
]

## Generate Conversations

In [ ]:
conversations = []

for i in range(NUM_CONVERSATIONS):
    num_turns = random.randint(5, 15) * 2  # total turns (both sides)
    scenario = SCENARIOS[i % len(SCENARIOS)]

    # Build prompt with scenario hint
    prompt = f"Scenario: {scenario}\n\n" + GENERATION_PROMPT_TEMPLATE.format(num_turns=num_turns)

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=2048,
            temperature=0.8,
            top_p=0.95,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated_text = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    # Prepend "agent(u1):" since the prompt ended right before it
    full_transcript = "agent(u1):" + generated_text.strip()

    conversations.append({
        "conversation_id": f"fake_{i:03d}",
        "label": 1,
        "transcript": full_transcript,
    })

    print(f"[{i+1}/{NUM_CONVERSATIONS}] Generated conversation fake_{i:03d} "
          f"(target ~{num_turns} turns)")

print(f"\nDone. Generated {len(conversations)} conversations.")

## Parse & Validate

Verify each generated transcript follows the expected `agent(uN): ...` / `customer(uN): ...` format.

In [ ]:
TURN_PATTERN = re.compile(r'^(agent|customer)\(u(\d+)\):\s*(.+)$', re.MULTILINE)

print(f"{'ID':<12} {'Turns':>6}  {'Alternating':>12}  First line preview")
print("-" * 80)

for conv in conversations:
    turns = TURN_PATTERN.findall(conv["transcript"])
    n_turns = len(turns)

    # Check strict alternation: agent, customer, agent, customer, ...
    alternating = True
    for j, (speaker, idx, text) in enumerate(turns):
        expected_speaker = "agent" if j % 2 == 0 else "customer"
        if speaker != expected_speaker:
            alternating = False
            break

    first_line = turns[0][2][:60] + "..." if turns and len(turns[0][2]) > 60 else (turns[0][2] if turns else "N/A")
    status = "OK" if alternating and n_turns >= 4 else "WARN"

    print(f"{conv['conversation_id']:<12} {n_turns:>6}  {status:>12}  {first_line}")

    if n_turns < 4:
        print(f"  ⚠ Too few turns — may need regeneration")

## (Optional) Repair Malformed Transcripts
If any transcripts have broken formatting, this cell attempts to fix common issues:\n- Remove blank lines between turns\n- Fix utterance numbering gaps\n- Drop non-transcript text before/after the conversation

In [ ]:
def repair_transcript(raw_text: str) -> str:
    """Extract valid turns and re-number them sequentially."""
    turns = TURN_PATTERN.findall(raw_text)
    if not turns:
        return raw_text  # nothing to repair

    repaired_lines = []
    for j, (speaker, _old_idx, text) in enumerate(turns):
        new_idx = j + 1
        # Force strict alternation
        expected_speaker = "agent" if j % 2 == 0 else "customer"
        repaired_lines.append(f"{expected_speaker}(u{new_idx}): {text.strip()}")

    return "\n".join(repaired_lines)


for conv in conversations:
    conv["transcript"] = repair_transcript(conv["transcript"])

# Re-validate after repair
for conv in conversations:
    turns = TURN_PATTERN.findall(conv["transcript"])
    assert len(turns) >= 4, f"{conv['conversation_id']} has only {len(turns)} turns after repair"
    for j, (speaker, idx, text) in enumerate(turns):
        expected = "agent" if j % 2 == 0 else "customer"
        assert speaker == expected, f"{conv['conversation_id']} turn {j}: expected {expected}, got {speaker}"
        assert int(idx) == j + 1, f"{conv['conversation_id']} turn {j}: expected u{j+1}, got u{idx}"

print("All conversations pass validation after repair.")

## Convert to Training Examples (same logic as notebook 01)

Parse the generated transcripts into the special-token format that notebook 02
expects. Each substantive agent turn becomes one training example with `prefix`
and `target` fields for loss masking.

In [ ]:
# ---------- constants (must match notebook 02) ----------
SYSTEM_PROMPT = (
    "You are an outbound sales agent for American Express.\n"
    "Your goal is to identify customer needs and guide the conversation\n"
    "toward [product/outcome]. Use information provided in the context\n"
    "block when available to reference relevant offers, campaigns,\n"
    "or product details. If no context is provided, rely on the\n"
    "conversation history to guide your response."
)

CONTEXT_TEMPLATES = [
    "Product: Corporate Platinum Card\nCampaign: Q2 2025 Spend Bonus",
    "Product: Business Gold Card\nCampaign: New Business Welcome Offer",
    "Product: Corporate Green Card\nCampaign: Annual Fee Waiver Promo",
    "Product: Business Platinum Card\nOffer: 150K MR Points Sign-Up Bonus",
    "Product: Blue Business Plus Card\nCampaign: 0% Intro APR for 12 Months",
]


# ---------- helpers ----------
def parse_turns(transcript: str) -> list[dict]:
    matches = TURN_PATTERN.findall(transcript)
    return [{"speaker": s, "index": int(i), "text": t.strip()} for s, i, t in matches]


def is_substantive(text: str) -> bool:
    return len(tokenizer.encode(text, add_special_tokens=False)) >= MIN_AGENT_TOKENS


def window_turns(turns: list[dict], target_idx: int) -> list[dict]:
    history = turns[:target_idx]
    if len(history) <= WINDOW_SIZE_K:
        return history
    return history[:2] + history[-(WINDOW_SIZE_K - 2):]


def build_example(turns: list[dict], target_pos: int, conv_id: str) -> dict:
    history = window_turns(turns, target_pos)
    target_text = turns[target_pos]["text"]

    context = random.choice(CONTEXT_TEMPLATES) if random.random() < CONTEXT_POPULATE_RATE else ""

    conv_lines = []
    for t in history:
        marker = "<|agent|>" if t["speaker"] == "agent" else "<|customer|>"
        conv_lines.append(f"{marker}{t['text']}")

    prefix = (
        f"<|system|>{SYSTEM_PROMPT}<|/system|>\n"
        f"<|context|>{context}<|/context|>\n"
        f"<|conversation|>\n"
        + "\n".join(conv_lines) + "\n"
        f"<|/conversation|>\n"
        f"<|agent|>"
    )

    return {
        "conversation_id": conv_id,
        "target_turn_index": turns[target_pos]["index"],
        "prefix": prefix,
        "target": target_text,
        "full_text": prefix + target_text,
    }


# ---------- build all training examples ----------
all_examples = []
skipped = 0

for conv in conversations:
    turns = parse_turns(conv["transcript"])
    for i, turn in enumerate(turns):
        if turn["speaker"] != "agent":
            continue
        if not is_substantive(turn["text"]):
            skipped += 1
            continue
        all_examples.append(build_example(turns, i, conv["conversation_id"]))

print(f"Generated {len(all_examples)} training examples from {len(conversations)} conversations")
print(f"Skipped {skipped} trivial agent turns (< {MIN_AGENT_TOKENS} tokens)")

# ---------- save ----------
OUTPUT_DIR.mkdir(exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for ex in all_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"\nSaved to {OUTPUT_FILE}")

# ---------- preview ----------
ex = all_examples[0]
print(f"\n{'='*60}")
print(f"Preview: {ex['conversation_id']}, target turn u{ex['target_turn_index']}")
print(f"{'='*60}")
print("PREFIX (last 300 chars):")
print(ex["prefix"][-300:])
print(f"\nTARGET:\n{ex['target']}")